# Ma Place · ce que disent les données de septembre 2026

Ce notebook utilise uniquement la base SQLite fictive du repo. Exécutez les cellules dans l'ordre. Les graphiques montrent des **observations**, pas des causes certaines. Consultez `SCHEMA.md` pour les définitions.

**Contrôle avant toute conclusion :** une ligne enregistrée peut être un second envoi de la même réservation. Les comptes de service sont une seconde question de qualité, indépendante du canal `api` lui-même.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

source = Path("ma-place-2026-09.db")
if not source.exists():
    source = Path("analytics/ma-place-2026-09.db")
assert source.exists(), "Base absente : recréez le repo depuis le modèle du cours."
conn = sqlite3.connect(f"file:{source.resolve()}?mode=ro", uri=True)
bookings = pd.read_sql_query("SELECT * FROM bookings ORDER BY booking_id", conn)
checkins = pd.read_sql_query("SELECT booking_id FROM checkins", conn)
rooms = pd.read_sql_query("SELECT * FROM rooms", conn)
desks = pd.read_sql_query("SELECT * FROM desks", conn)
people = pd.read_sql_query("SELECT * FROM people", conn)
badges = pd.read_sql_query("SELECT person_id FROM badge_entries", conn)
events = pd.read_sql_query("SELECT * FROM app_events", conn)
print(f"Lignes brutes de réservations : {len(bookings):,}")

## Contrôle 1 · de quoi compte-t-on les lignes ?

Conservez le premier `booking_id` d'une réservation identique. Un second envoi n'est pas une seconde place occupée. Comparez le résultat à **1 671 réservations distinctes** ; si le nombre diffère, vérifiez la clé et l'ordre.

In [ ]:
cle = ["person_id", "kind", "resource_id", "day", "start_time", "end_time"]
uniques = bookings.drop_duplicates(subset=cle, keep="first").copy()
print(f"Lignes brutes : {len(bookings):,}")
print(f"Envois répétés : {len(bookings) - len(uniques):,}")
print(f"Contrôle 1 — réservations distinctes : {len(uniques):,}")
assert len(uniques) == 1671, "Le contrôle 1 ne correspond pas : revoyez la déduplication."

# Niveau 2 : les comptes dont toutes les réservations passent par l'API à 03 h
# et qui n'ont jamais badgé. Une vraie personne peut aussi utiliser l'API.
sans_badge = set(people.person_id) - set(badges.person_id)
comptes_service = [person for person, groupe in bookings.groupby("person_id")
                   if person in sans_badge and (groupe.channel == "api").all()
                   and groupe.created_at.str[11:13].eq("03").all()]
analyse = uniques.loc[~uniques.person_id.isin(comptes_service)].copy()
print(f"Après exclusion des comptes de service : {len(analyse):,} réservations ; "
      f"{analyse.person_id.nunique()} personnes")

## 1 · Réunions fantômes

Une salle réservée sans check-in peut être restée vide, mais l'absence de confirmation ne le prouve pas. Le graphique compare les salles ; le taux global garde **toutes** les salles au dénominateur.

In [ ]:
reunions = analyse.loc[analyse.kind == "room"].merge(
    checkins.assign(confirmee=True), on="booking_id", how="left")
reunions["fantome"] = reunions.confirmee.isna()
par_salle = (reunions.groupby("resource_id").fantome.agg(["sum", "count", "mean"])
             .join(rooms.set_index("room_id")["name"]).sort_values("mean"))
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(par_salle["name"], 100 * par_salle["mean"], color="#365b88")
ax.set(xlim=(0, 100), xlabel="Réservations sans check-in (%)", title="Réunions fantômes par salle")
ax.grid(axis="x", alpha=.2)
plt.tight_layout(); plt.show()
print(f"Toutes salles : {reunions.fantome.sum():,} / {len(reunions):,} "
      f"= {reunions.fantome.mean():.1%}")

## 2 · Une salle de huit places pour deux personnes ?

La taille annoncée ne dit rien du matériel, de l'accessibilité ou des autres besoins. Une suggestion de salle plus petite doit rester facultative.

In [ ]:
grandes = reunions.merge(rooms[["room_id", "capacity"]],
                          left_on="resource_id", right_on="room_id")
grands_creneaux = grandes.loc[grandes.capacity >= 8]
petits_groupes = int((grands_creneaux.attendees <= 2).sum())
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["1–2 personnes", "3 personnes ou plus"],
       [petits_groupes, len(grands_creneaux) - petits_groupes],
       color=["#b56935", "#365b88"])
ax.set(ylabel="Réservations", title="Salles de huit places ou plus")
ax.grid(axis="y", alpha=.2)
plt.tight_layout(); plt.show()
print(f"{petits_groupes:,} / {len(grands_creneaux):,} "
      f"= {petits_groupes / len(grands_creneaux):.1%}")

## 3 · Postes par jour de semaine et par équipe

Calculez d'abord les **postes distincts par date**, puis la moyenne par jour de semaine. Compter les lignes brutes peut produire plus de 100 % d'occupation.

In [ ]:
postes = analyse.loc[analyse.kind == "desk"].copy()
postes["date"] = pd.to_datetime(postes.day)
quotidien = postes.groupby("date").resource_id.nunique().div(len(desks))
jours = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi"]
moyenne = quotidien.groupby(quotidien.index.dayofweek).mean().reindex(range(5))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(jours, 100 * moyenne, color="#3f806e")
ax.set(ylim=(0, 100), ylabel="Postes distincts réservés (%)",
       title="Occupation moyenne selon le jour")
ax.grid(axis="y", alpha=.2)
plt.tight_layout(); plt.show()
print(dict(zip(jours, (100 * moyenne).round(1))))
zones = postes.merge(people, on="person_id").merge(
    desks, left_on="resource_id", right_on="desk_id")
print("Dans la zone de leur équipe (% des réservations) :")
print((100 * zones.assign(meme_zone=zones.team == zones.zone)
       .groupby("team").meme_zone.mean()).round(1).to_string())

## 4 · Erreurs dans l'application et demande d'agenda

Regardez l'heure des erreurs, puis comptez séparément les clics et les personnes qui ont cliqué sur le lien désactivé. Une concentration horaire ne démontre pas à elle seule la cause d'un bug.

In [ ]:
erreurs = events.loc[events.event == "booking_error"].copy()
erreurs["heure"] = erreurs.ts.str[11:13].astype(int)
par_heure = erreurs.groupby("heure").size().reindex(range(7, 20), fill_value=0)
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(par_heure.index, par_heure.values,
       color=["#b56935" if h < 10 else "#365b88" for h in par_heure.index])
ax.set(xticks=list(par_heure.index), xlabel="Heure", ylabel="Événements booking_error",
       title="Erreurs de réservation selon l'heure")
ax.grid(axis="y", alpha=.2)
plt.tight_layout(); plt.show()
agenda = events.loc[events.event == "agenda_click_disabled"]
cliqueurs = agenda.loc[agenda.person_id.isin(analyse.person_id)].person_id.nunique()
print(f"Lien agenda désactivé : {len(agenda):,} clics, "
      f"{cliqueurs} personnes parmi {analyse.person_id.nunique()} ayant réservé")
conn.close()

## Décisions à inscrire dans le backlog

Pour chacune des quatre observations, rédigez une carte avec un but, **un chiffre accompagné de son dénominateur** et une vérification observable. Les cartes A–D du `BACKLOG.md` au point `w4-analyses` montrent un résultat possible. Vérifiez les nombres avant de les recopier ; ne donnez jamais de lignes individuelles à Claude.